# Ground Truth : palier 3 sur un GPU T4 (le corpus bouge)

Ce notebook est aussi la **répétition générale de la finale** : il lance exactement les commandes de
`run.sh` que l'orga exécutera (`index`, puis `update`, puis `answer`) et les chronomètre.
**Avant de commencer :** Exécution → Modifier le type d'exécution → **GPU T4**. Garde le PC branché, sans veille.

| Étape | Durée | Produit |
|---|---|---|
| 0. Installation + Drive | 10 min | dépôt, modèles, corpus v1 et v2 |
| 1. `run.sh index` puis `run.sh update` | 35 min | index v1 (OCR français) puis v1 + v2, chronométrés |
| 2. Validation v2 | 20 min | ablation de la recherche, score, échecs commentables |
| 3. Validation v1 avec l'OCR français (facultatif) | 20 min | effet de l'OCR français sur les scans |
| 4. Soumission v2 + rapatriement | 20 min | `submissions/v2/…json`, `a_rapatrier_p3.zip` |

Chaque étape longue se termine par une **sauvegarde sur le Drive** : si la session coupe, on reprend là.

## 0. Installation

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
REPO = "https://github.com/YanisYounsi-code/Ground-Truth-un-RAG-qui-ne-ment-pas.git"
!rm -rf /content/gt && git clone -q $REPO /content/gt
%cd /content/gt
!git log --oneline -3

In [ ]:
# Dépendances, Tesseract français, poids des modèles, corpus v1 (110 PDF) et v2 (20 PDF) : environ 10 minutes
!bash setup.sh
!python corpus/download_corpus.py
!ls corpus/pdf | wc -l; ls corpus/pdf_v2 | wc -l
!tesseract --list-langs | grep -x fra && echo "OCR français OK"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/gt_sauvegarde/p3

## 1. Répétition générale : `run.sh index` puis `run.sh update`

Limites officielles : **2 h** pour l'index v1, **20 min** pour la mise à jour v2, sans réindexer la v1.
On garde une copie de l'index avant la mise à jour (`/content/idx/v1fr`) pour l'étape 3.

In [ ]:
!time bash run.sh index corpus/pdf /content/idx/final
!cp -r /content/idx/final /content/idx/v1fr
!cp -r /content/idx/v1fr /content/drive/MyDrive/gt_sauvegarde/p3/

In [ ]:
!time bash run.sh update /content/idx/final corpus/pdf_v2
!python -c "import json; c=json.load(open('/content/idx/final/chunks.json')); print(len(c), 'chunks,', len({x['doc_id'] for x in c}), 'documents')"
!cp -r /content/idx/final /content/drive/MyDrive/gt_sauvegarde/p3/

**Si la session a coupé** après l'étape 1 : relance les 3 cellules de l'étape 0, puis celle-ci, et reprends à l'étape 2.

In [ ]:
!mkdir -p /content/idx && cp -r /content/drive/MyDrive/gt_sauvegarde/p3/final /content/drive/MyDrive/gt_sauvegarde/p3/v1fr /content/idx/ && ls /content/idx
%cd /content/gt

## 2. Validation v2 (40 questions : temporelles, en anglais, multi-doc, synthèses)

**2.1 Recherche** : le pipeline complet sur l'index v1 + v2, une brique retirée à la fois, et l'index
**v1 seul** pour montrer ce qu'apporte la mise à jour (beaucoup de preuves sont dans les documents v2).

In [ ]:
!python -m pipeline.ablation --questions data/validation_v2.json --out des_tests/ablation_v2.md \
    --config complet=/content/idx/final:hybrid:rerank \
    --config sans_reranker=/content/idx/final:hybrid:norerank \
    --config sans_dense=/content/idx/final:bm25:rerank \
    --config sans_bm25=/content/idx/final:dense:rerank \
    --config index_v1_seul=/content/idx/v1fr:hybrid:rerank

**2.2 Score** : `run.sh answer` exactement comme à la finale (chronométré), puis le juge officiel et les échecs.

In [ ]:
!time bash run.sh answer /content/idx/final data/validation_v2.json runs/val2.json
!python -m groundtruth.evaluate --pred runs/val2.json --gold data/validation_v2.json --judge hf --out runs/val2_report.json
!python -m pipeline.failures --pred runs/val2.json --report runs/val2_report.json --gold data/validation_v2.json --out des_tests/echecs_v2.md --n 8
!mkdir -p /content/drive/MyDrive/gt_sauvegarde/p3/runs && cp runs/val2* /content/drive/MyDrive/gt_sauvegarde/p3/runs/ && cp des_tests/ablation_v2.md des_tests/echecs_v2.md /content/drive/MyDrive/gt_sauvegarde/p3/

## 3. Validation v1 avec l'OCR français (facultatif, 20 min)

Le score +0,38 du palier 2 a été obtenu avec des scans lus en anglais. Même pipeline, index v1 refait
avec l'OCR français : la différence mesure l'effet de la langue de l'OCR (surtout sur les 9 questions scan).

In [ ]:
!bash run.sh answer /content/idx/v1fr data/validation_v1.json runs/val1_fr.json
!python -m groundtruth.evaluate --pred runs/val1_fr.json --gold data/validation_v1.json --judge hf --out runs/val1_fr_report.json
!cp runs/val1_fr* /content/drive/MyDrive/gt_sauvegarde/p3/runs/

## 4. Soumission v2 et rapatriement

In [ ]:
!time bash run.sh answer /content/idx/final data/classement_v2_questions.json runs/classement_v2.json
!python -m groundtruth.submit --pred runs/classement_v2.json --phase v2

In [ ]:
# Seulement les fichiers nouveaux : ne pas écraser des_tests/ de ton PC, plus récent que celui de Colab
!cp -r runs submissions /content/drive/MyDrive/gt_sauvegarde/p3/
!rm -rf /content/rap && mkdir -p /content/rap/des_tests/rapports
!cp runs/*_report.json /content/rap/des_tests/rapports/ && cp runs/generator_log.jsonl /content/rap/des_tests/rapports/generator_log_p3.jsonl
!cp --parents submissions/v2/*.json des_tests/ablation_v2.md des_tests/echecs_v2.md /content/rap/
!cd /content/rap && zip -q -r /content/a_rapatrier_p3.zip . && unzip -l /content/a_rapatrier_p3.zip
from google.colab import files; files.download("/content/a_rapatrier_p3.zip")